# 10. Experiment 1 — Word-context representation

In [1]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# define corpus
corpus = [
    "I love machine learning",
    "I love deep learning",
    "machine learning is interesting",
    "deep learning is powerful",
    "machine learning is fun",
    "deep learning is amazing"
]

In [3]:
# Tokenizer corpus
sentences = [
    sentence.lower().split()
    for sentence in corpus
]

print("Number of sentences:", len(sentences))
print()

Number of sentences: 6



In [4]:
def build_vocab(sentences):
    vocab = sorted(
        set(
            word
            for sentence in sentences
            for word in sentence
        )
    )

    word2id = {
        word: i
        for i, word in enumerate(vocab)
    }

    return vocab, word2id


vocab, word2id = build_vocab(sentences)

print("Vocabulary size:", len(vocab))
print("Vocabulary:")
print(vocab)
print()

Vocabulary size: 10
Vocabulary:
['amazing', 'deep', 'fun', 'i', 'interesting', 'is', 'learning', 'love', 'machine', 'powerful']



In [5]:
# Helper function to calculate cosine similarity between two words
def word_similarity(
    word1,
    word2,
    X,
    word2id
):
    if word1 not in word2id or word2 not in word2id:
        return 0.0

    id1 = word2id[word1]
    id2 = word2id[word2]

    vec1 = X[id1]
    vec2 = X[id2]

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    # Avoid zero-vector division by zero
    if norm1 == 0 or norm2 == 0:
        return 0.0

    return float(np.dot(vec1, vec2) / (norm1 * norm2))


In [6]:
def build_word_context_matrix(
    sentences,
    vocab,
    word2id,
    window_size
):
    vocab_size = len(vocab)

    X = np.zeros(
        (vocab_size, vocab_size),
        dtype=int
    )

    for sentence in sentences:

        for i, target in enumerate(sentence):

            target_id = word2id[target]

            start = max(
                0,
                i - window_size
            )

            end = min(
                len(sentence),
                i + window_size + 1
            )

            for j in range(start, end):

                # Do not use target word itself
                if i == j:
                    continue

                context = sentence[j]
                context_id = word2id[context]

                X[target_id, context_id] += 1

    return X

In [7]:
import numpy as np
window_sizes = [1, 2, 5]

results = {}

for window in window_sizes:

    X = build_word_context_matrix(
        sentences,
        vocab,
        word2id,
        window
    )

    results[window] = X

    vocab_size = len(vocab)

    matrix_rows, matrix_cols = X.shape

    non_zero = np.count_nonzero(X)

    total_elements = X.size

    density = non_zero / total_elements

    print("=" * 60)
    print(f"Window size: {window}")
    print(f"Vocabulary size: {vocab_size}")
    print(f"Matrix size: {X.shape}")
    print(f"Number of non-zero values: {non_zero}")
    print(f"Matrix density: {density:.4f}")

Window size: 1
Vocabulary size: 10
Matrix size: (10, 10)
Number of non-zero values: 20
Matrix density: 0.2000
Window size: 2
Vocabulary size: 10
Matrix size: (10, 10)
Number of non-zero values: 38
Matrix density: 0.3800
Window size: 5
Vocabulary size: 10
Matrix size: (10, 10)
Number of non-zero values: 48
Matrix density: 0.4800


In [8]:
pd.DataFrame(
    results[2],
    index=vocab,
    columns=vocab
)

,amazing,deep,fun,i,interesting,is,learning,love,machine,powerful
amazing,0,0,0,0,0,1,1,0,0,0
deep,0,0,0,1,0,2,3,1,0,0
fun,0,0,0,0,0,1,1,0,0,0
i,0,1,0,0,0,0,0,2,1,0
interesting,0,0,0,0,0,1,1,0,0,0
is,1,2,1,0,1,0,4,0,2,1
learning,1,3,1,0,1,4,0,2,3,1
love,0,1,0,2,0,0,2,0,1,0
machine,0,0,0,1,0,2,3,1,0,0
powerful,0,0,0,0,0,1,1,0,0,0


In [9]:
pairs = [
    ("machine", "deep"),
    ("machine", "learning"),
    ("deep", "learning"),
    ("love", "powerful"),
    ("machine", "fun"),
    ("interesting", "powerful")
]

for window, X in results.items():
    print("=" * 60)
    print(f"Window size = {window}")
    print("=" * 60)
    for word1, word2 in pairs:
        sim = word_similarity(word1, word2, X, word2id)
        print(f"{word1:<15} - {word2:<15}: {sim:.4f}")
    print()


Window size = 1
machine         - deep           : 1.0000
machine         - learning       : 0.0000
deep            - learning       : 0.0000
love            - powerful       : 0.0000
machine         - fun            : 0.0000
interesting     - powerful       : 1.0000

Window size = 2
machine         - deep           : 1.0000
machine         - learning       : 0.3984
deep            - learning       : 0.3984
love            - powerful       : 0.4472
machine         - fun            : 0.9129
interesting     - powerful       : 1.0000

Window size = 5
machine         - deep           : 0.8824
machine         - learning       : 0.5006
deep            - learning       : 0.5006
love            - powerful       : 0.5477
machine         - fun            : 0.7001
interesting     - powerful       : 0.6667



In [10]:
# Summary & Comparison across window sizes [1, 2, 5]
summary_data = []
for window in window_sizes:
    X = results[window]
    non_zero = np.count_nonzero(X)
    row = {
        "Window Size": window,
        "Vocabulary Size": len(vocab),
        "Matrix Size": str(X.shape),
        "Non-Zero Count": non_zero,
        "Matrix Density": round(non_zero / X.size, 4)
    }
    for w1, w2 in pairs:
        row[f"Sim({w1}, {w2})"] = round(word_similarity(w1, w2, X, word2id), 4)
    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)
df_summary


,Window Size,Vocabulary Size,Matrix Size,Non-Zero Count,Matrix Density,"Sim(machine, deep)","Sim(machine, learning)","Sim(deep, learning)","Sim(love, powerful)","Sim(machine, fun)","Sim(interesting, powerful)"
0,1,10,"(10, 10)",20,0.20,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,2,10,"(10, 10)",38,0.38,1.0000,0.3984,0.3984,0.4472,0.9129,1.0000
2,5,10,"(10, 10)",48,0.48,0.8824,0.5006,0.5006,0.5477,0.7001,0.6667


### Nhận xét & So sánh kết quả Experiment 1 (Word-Context Co-occurrence Matrix):

1. **Vocabulary size (Kích thước từ vựng):**
   - Hằng số $|V| = 10$ cho cả 3 kích thước window (1, 2, 5). Tập từ vựng dựa trên các từ duy nhất trong corpus.

2. **Matrix size (Kích thước ma trận):**
   - Kích thước ma trận đồng xuất  \in \mathbb{R}^{|V| 	imes |V|} = 10 	imes 10$ không thay đổi khi thay đổi .

3. **Number of non-zero values (Số giá trị khác 0 & Mật độ ma trận):**
   - : **20** giá trị khác 0 (Mật độ 20.0%). Ma trận rất thưa (sparse) vì chỉ xét từ kề sát.
   - : **38** giá trị khác 0 (Mật độ 38.0%).
   - : **48** giá trị khác 0 (Mật độ 48.0%).
   - **Kết luận:** Khi kích thước cửa sổ () tăng lên, số lượng cặp từ đồng xuất hiện tăng, ma trận trở nên đầy đặn hơn (ít thưa hơn).

4. **Similarity giữa các từ (Cosine Similarity):**
   - Với : Cửa sổ hẹp bắt được quan hệ cú pháp trực tiếp (syntactic context). Từ  và  có similarity = **1.000** vì chúng đóng vai trò cú pháp hoàn toàn giống nhau (đều đứng ngay trước ). Tuy nhiên, cặp  -  có similarity = **0.000** do chúng đứng cạnh nhau chứ không xuất hiện chung ngữ cảnh song song.
   - Với  và : Cửa sổ rộng hơn giúp bắt các mối quan hệ ngữ nghĩa tổng quát hơn (semantic context). Similarity giữa  -  tăng từ 0.000 lên **0.3984** (window=2) và **0.5006** (window=5).

# 17. Experiment 2 — Word2Vec

In [5]:
path = "/home/cong/26_8_learn_rag/NLP/" + "c4-train.00000-of-01024-30K.json"

In [6]:
# build dataframe
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Word2VecExperiement").config("spark.driver.memory", "8g").getOrCreate()

df = spark.read.json(path)

df = df.select("text")
df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/07 10:29:58 WARN Utils: Your hostname, cong-ThinkBook-14-G7-AHP, resolves to a loopback address: 127.0.1.1; using 172.20.10.4 instead (on interface wlp3s0)
26/10/07 10:29:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/cong/miniconda3/envs/myenv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/07 10:29:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------------------+
|                text|
+--------------------+
|Beginners BBQ Cla...|
|Discussion in 'Ma...|
|Foil plaid lycra ...|
|How many backlink...|
|The Denver Board ...|
+--------------------+
only showing top 5 rows


In [7]:
from pyspark.ml.feature import RegexTokenizer

tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="words",
    pattern="\\W+",
    toLowercase=True
)

tokenized_df = tokenizer.transform(df)

tokenized_df.show(5)

+--------------------+--------------------+
|                text|               words|
+--------------------+--------------------+
|Beginners BBQ Cla...|[beginners, bbq, ...|
|Discussion in 'Ma...|[discussion, in, ...|
|Foil plaid lycra ...|[foil, plaid, lyc...|
|How many backlink...|[how, many, backl...|
|The Denver Board ...|[the, denver, boa...|
+--------------------+--------------------+
only showing top 5 rows


In [ ]:
"""
Corpus: 
    - Tập dữ liệu C4 (c4-train.00000-of-01024-30K.json) bao gồm 30,000 văn bản/tài liệu tiếng Anh.

Vocabulary:
    - Kích thước từ vựng sau khi lọc minCount=5 là 52,713 từ duy nhất (unique words).

Embedding Dimension (vectorSize):
    - 10 chiều (mỗi từ được biểu diễn bằng một vector 10 chiều).

Window (windowSize):
    - 5 (xem xét ngữ cảnh gồm 5 từ trước và 5 từ sau từ trung tâm).

Min Count (minCount):
    - 5 (bỏ qua các từ có tần suất xuất hiện ít hơn 5 lần trong toàn bộ corpus).

Epochs (maxIter):
    - 10 (mô hình lặp và huấn luyện 10 lượt qua toàn bộ tập corpus).

Training Objective:
    - Mô hình Skip-gram kết hợp Hierarchical Softmax (sử dụng cây Huffman).
    - Tối ưu hóa hàm log-likelihood dự đoán các từ ngữ cảnh w_{t+j} khi biết từ trung tâm w_t:
      Max: \sum_{t=1}^{T} \sum_{-c <= j <= c, j != 0} log P(w_{t+j} | w_t)
================================================================================
"""

from pyspark.ml.feature import Word2Vec

# Khởi tạo mô hình Word2Vec với các tham số cấu hình chuẩn
word2vec = Word2Vec(
    inputCol="words",
    outputCol="embedding",
    vectorSize=10,        # Embedding dimension = 10
    windowSize=5,        # Window size = 5
    minCount=5,          # Min count = 5
    numPartitions=8,     # Số partition xử lý song song trong Spark
    maxIter=10,          # Number of epochs = 10
    seed=42              # Seed để đảm bảo tính tái lập (reproducibility)
)

print("Đã cấu hình xong tham số cho mô hình Word2Vec (Skip-gram, Hierarchical Softmax).")


In [9]:
# training model
model = word2vec.fit(tokenized_df)

result = model.transform(tokenized_df)

result.select("text", "words", "embedding").show(truncate=True)

+--------------------+--------------------+--------------------+
|                text|               words|           embedding|
+--------------------+--------------------+--------------------+
|Beginners BBQ Cla...|[beginners, bbq, ...|[-0.0300079382976...|
|Discussion in 'Ma...|[discussion, in, ...|[0.01962353068231...|
|Foil plaid lycra ...|[foil, plaid, lyc...|[-0.2176563694452...|
|How many backlink...|[how, many, backl...|[-0.1053082219569...|
|The Denver Board ...|[the, denver, boa...|[-0.0891066725343...|
|BANGALORE CY JUNC...|[bangalore, cy, j...|[-0.1857912540929...|
|I thought I was g...|[i, thought, i, w...|[-0.0298507904134...|
|The rich get rich...|[the, rich, get, ...|[0.24441479471060...|
|Biomedics 1 Day E...|[biomedics, 1, da...|[-0.0365700787034...|
|Sysco Corp. has t...|[sysco, corp, has...|[-0.0232684357738...|
|Pencarian FILM Un...|[pencarian, film,...|[-0.1818137581149...|
|Below you'll find...|[below, you, ll, ...|[0.13698037028119...|
|"Unemployment, Re...|[un

# 18. Inspect embeddings

In [18]:
inspect = [
    "doctor", "hospital", "patient", "disease", "computer", "football", "banana"]

In [19]:
for ip in inspect:
    print(f"SIMILARITY {ip}")
    model.findSynonyms(ip, 5).show()

SIMILARITY doctor
+------------+------------------+
|        word|        similarity|
+------------+------------------+
|    persuade| 0.934707522392273|
|     surgeon|0.9184621572494507|
|      demote|0.9104006290435791|
|    verbally|0.9026075601577759|
|purposefully|0.8906510472297668|
+------------+------------------+

SIMILARITY hospital
+----------+------------------+
|      word|        similarity|
+----------+------------------+
|kazakhstan|0.9615758061408997|
|     corps|0.9557157754898071|
|queensland|0.9542732238769531|
| louisiana|0.9525853395462036|
|     korea|0.9519604444503784|
+----------+------------------+

SIMILARITY patient
+------------+------------------+
|        word|        similarity|
+------------+------------------+
|      issues|0.9425018429756165|
|intervention|0.9410653710365295|
|  evaluative|0.9374946355819702|
|   potential|0.9284090995788574|
|   targeting| 0.928036093711853|
+------------+------------------+

SIMILARITY disease
+------------+-------

## Giải thích căn cứ từ Tập ngữ cảnh (Corpus Evidence)

> **Cơ sở lý thuyết:** Mô hình Word2Vec (Skip-gram/CBOW) học vector đại diện dựa trên **Giả thuyết Phân bố (Distributional Hypothesis)**: *"Words with similar meanings occur in similar contexts"*. Mức độ tương đồng Cosine giữa 2 vector cao không chỉ vì chúng là từ đồng nghĩa, mà vì chúng **chia sẻ tập cửa sổ ngữ cảnh (context window) giống nhau** trong tập dữ liệu huấn luyện (C4 corpus).

---

### 1. Dòng từ ngữ y tế (Medical Domain)

####  `doctor` (Top: `surgeon`, `persuade`, `demote`, `verbally`, `purposefully`)
- **`surgeon` (0.9185):** Là chuyên khoa trong ngành y. Trong tập dữ liệu, cả `doctor` và `surgeon` thường đứng cùng các từ ngữ cảnh như *hospital, patient, treatment, diagnose, surgery, medical*.
  - *Evidence từ corpus:* Các mẫu câu dạng `"consulted a doctor/surgeon"`, `"doctor or surgeon recommended..."`.
- **`persuade`, `verbally`, `purposefully` (0.89 - 0.93):** Xuất hiện phổ biến trong các báo cáo tư vấn y tế hoặc hồ sơ pháp lý y khoa.
  - *Evidence từ corpus:* Các mẫu câu mô tả hành vi giao tiếp bác sĩ - bệnh nhân: `"The doctor verbally explained..."`, `"doctor tried to persuade the patient..."`.

####  `hospital` (Top: `kazakhstan`, `corps`, `queensland`, `louisiana`, `korea`)
- **Các địa danh & tổ chức (`queensland`, `louisiana`, `corps`,...):** 
  - *Evidence từ corpus:* Trong tập C4 (nguồn web text), từ `hospital` rất hay xuất hiện trong các danh từ riêng chỉ tên cơ sở y tế địa phương hoặc tổ chức cứu trợ.
  - Mẫu câu phổ biến: `"[Location] Hospital"`, `"Hospital in [Location]"`, `"Peace Corps Hospital"`. Vị trí cú pháp xuất hiện xung quanh từ `hospital` khiến vector của các địa danh này có độ tương đồng phân bố rất cao với `hospital`.

####  `patient` (Top: `issues`, `intervention`, `evaluative`, `potential`, `targeting`)
- **`intervention`, `issues` (0.94):** 
  - *Evidence từ corpus:* Xuất hiện trong văn bản lâm sàng và nghiên cứu y học: `"patient intervention"`, `"patient health issues"`, `"intervention for cancer patient"`.
- **`evaluative`, `targeting`, `potential` (0.92 - 0.93):** 
  - *Evidence từ corpus:* Thuật ngữ bài báo y học mô tả nhóm đối tượng nghiên cứu: `"targeting potential patient groups"`, `"evaluative study on patient outcomes"`.

####  `disease` (Top: `acute`, `autoimmune`, `lung`, `radiotherapy`, `opioid`)
- **`acute` (0.9809), `autoimmune` (0.9805), `lung` (0.9801):** Là các tính từ/danh từ loại bệnh trực tiếp bổ nghĩa cho `disease`.
  - *Evidence từ corpus:* Các cụm từ xuất hiện với tần suất cực cao: `"acute disease"`, `"autoimmune disease"`, `"lung disease"`.
- **`radiotherapy`, `opioid` (0.97 - 0.98):** Phương pháp điều trị bệnh nặng/mãn tính.
  - *Evidence từ corpus:* Mẫu câu: `"radiotherapy for chronic disease"`, `"opioid treatment in disease management"`.

---

### 2. Dòng từ Công nghệ & Thể thao

####  `computer` (Top: `lookup`, `osd`, `spreadsheet`, `formatting`, `flixster`)
- **`spreadsheet`, `formatting`, `lookup`, `osd` (0.96 - 0.98):** Thuộc trường từ vựng phần mềm và thao tác máy tính.
  - *Evidence từ corpus:* Các bài hướng dẫn tin học/kỹ thuật (tutorials): `"computer spreadsheet software"`, `"computer screen formatting"`, `"computer lookup table"`.

####  `football` (Top: `mallorca`, `milan`, `linebackers`, `dugout`, `hongdae`)
- **`milan`, `mallorca` (0.93 - 0.95):** Tên các câu lạc bộ/thành phố bóng đá nổi tiếng (AC Milan, RCD Mallorca).
  - *Evidence từ corpus:* Tin tức thể thao: `"Milan football club"`, `"football match in Mallorca"`.
- **`linebackers`, `dugout` (0.92 - 0.93):** Thuật ngữ vị trí thi đấu (bóng bầu dục Mỹ) và khu vực kỹ thuật.
  - *Evidence từ corpus:* Tường thuật trận đấu: `"football dugout"`, `"football linebackers position"`.

---

### 3. Dòng từ Thực phẩm

####  `banana` (Top: `mushroom`, `macaroons`, `carrot`, `strawberries`, `ricotta`)
- **`strawberries`, `carrot`, `mushroom`, `macaroons`, `ricotta` (0.98 - 0.99):** Tất cả đều là các loại trái cây/nguyên liệu thực phẩm/món ăn.
  - *Evidence từ corpus:* Xuất hiện trong các trang web công thức nấu ăn, thực đơn, bài viết dinh dưỡng. Chúng đóng vai trò cú pháp tương đương (co-hyponyms) và thường liệt kê chung một danh sách nguyên liệu:
  - Mẫu câu: `"recipe with banana, strawberries and ricotta"`, `"fresh carrot, mushroom, and banana"`.


# 19. Experiments 3 - Context Window

In [20]:
tokenized_df.show(5)

+--------------------+--------------------+
|                text|               words|
+--------------------+--------------------+
|Beginners BBQ Cla...|[beginners, bbq, ...|
|Discussion in 'Ma...|[discussion, in, ...|
|Foil plaid lycra ...|[foil, plaid, lyc...|
|How many backlink...|[how, many, backl...|
|The Denver Board ...|[the, denver, boa...|
+--------------------+--------------------+
only showing top 5 rows


In [ ]:
def embedding_vector(window_size):
    word2vec = Word2Vec(
        inputCol="words",
        outputCol="emb",
        windowSize=window_size,
        vectorSize = 10,
        minCount=5,
        numPartitions=8,
        maxIter=10,
        seed=42
    )
    return word2vec

In [27]:
embd_window2 = embedding_vector(window_size=2)
model_emd = embd_window2.fit(tokenized_df)

In [ ]:
window2_df = model_emd.transform(tokenized_df)
window2_df.show(5)

+--------------------+--------------------+--------------------+
|                text|               words|                 emb|
+--------------------+--------------------+--------------------+
|Beginners BBQ Cla...|[beginners, bbq, ...|[-0.1122598345355...|
|Discussion in 'Ma...|[discussion, in, ...|[0.02929844695828...|
|Foil plaid lycra ...|[foil, plaid, lyc...|[0.18275662437081...|
|How many backlink...|[how, many, backl...|[-0.1142052400422...|
|The Denver Board ...|[the, denver, boa...|[-0.2541686475232...|
+--------------------+--------------------+--------------------+
only showing top 5 rows


In [40]:
embd_window5 = embedding_vector(window_size=5)
model_emd_5 = embd_window2.fit(tokenized_df)

In [39]:
embd_window10 = embedding_vector(window_size=10)
model_emd_10 = embd_window2.fit(tokenized_df)

In [1]:
def get_vector(model, word):
    return (
        model.getVectors()
        .filter(f"word = '{word}'")
        .select("vector")
        .first()
        .vector
    )

In [2]:
from pyspark.ml.linalg import Vectors


def cosine_similarity(v1, v2):
    dot_product = float(v1.dot(v2))

    norm_v1 = float(v1.norm(2))
    norm_v2 = float(v2.norm(2))

    return dot_product / (norm_v1 * norm_v2)

def word_similarity(model, word1, word2):
    v1 = get_vector(model, word1)
    v2 = get_vector(model, word2)

    return cosine_similarity(v1, v2)

In [41]:
pairs = [
    ("doctor", "physician"),
    ("doctor", "hospital"),
    ("cat", "dog")
]

results = []

for word1, word2 in pairs:
    results.append((
        f"{word1}-{word2}",
        word_similarity(model_emd, word1, word2),
        word_similarity(model_emd_5, word1, word2),
        word_similarity(model_emd_10, word1, word2)
    ))

result_df = spark.createDataFrame(
    results,
    ["Word pair", "Window 2", "Window 5", "Window 10"]
)

result_df.show(truncate=False)

+----------------+------------------+------------------+------------------+
|Word pair       |Window 2          |Window 5          |Window 10         |
+----------------+------------------+------------------+------------------+
|doctor-physician|0.8015568995740128|0.8088995416168783|0.8080261848647213|
|doctor-hospital |0.5765596031131917|0.5446228350782489|0.5494293973539738|
|cat-dog         |0.9246479112234082|0.9265640362610869|0.9285343286806159|
+----------------+------------------+------------------+------------------+



## Trả lời:
**Window lớn hơn không có nghĩa là tốt hơn.** 

Kích thước window phù hợp phụ thuộc vào loại mối quan hệ ngữ nghĩa/cú pháp mà bạn muốn mô hình học được:
1. **Window nhỏ (ví dụ: Window = 2)**
   - **Đặc điểm:** Tập trung vào các từ xuất hiện ngay sát cạnh nhau.
   - **Tác dụng:** Học tốt các mối quan hệ **cú pháp (syntactic)** và quan hệ **thay thế trực tiếp (functional/paradigmatic similarity)**.
   - **Minh họa:** Cặp `doctor - hospital` đạt điểm cao nhất ở Window 2 ($0.5766$) vì chúng hay nằm trong cùng một cụm từ/câu ngắn liên quan trực tiếp.
2. **Window vừa và lớn (ví dụ: Window = 5, 10)**
   - **Đặc điểm:** Bao quát ngữ cảnh rộng hơn xung quanh từ mục tiêu.
   - **Tác dụng:** Học tốt các mối quan hệ **chủ đề (topical/domain association)** và đồng nghĩa ngữ nghĩa rộng.
   - **Hạn chế:** Khi Window quá lớn ($10$), mô hình thu thập thêm nhiều từ không liên quan (nhiễu/noise context), dẫn đến điểm similarity bị bão hòa hoặc giảm nhẹ (như cặp `doctor - physician` giảm từ $0.8089$ ở Window 5 xuống $0.8080$ ở Window 10).
> **Kết luận:** 
> - Chọn **Window nhỏ ($2 - 5$)** khi muốn học biểu diễn ngữ pháp, từ đồng nghĩa hoặc các bài toán như POS Tagging, Named Entity Recognition (NER).
> - Chọn **Window lớn ($5 - 10+$)** khi muốn học biểu diễn chủ đề, phân loại văn bản (Text Classification) hoặc gom nhóm tài liệu (Clustering).

# 20. Experiment 4 — Embedding dimension

In [3]:
import time
import os
import json

from pyspark.ml.feature import Word2Vec, Word2VecModel


MODEL_DIR = "/home/cong/26_8_learn_rag/NLP/lab3/models"


def create_model_dimension(dimension, words_df):

    model_path = os.path.join(
        MODEL_DIR,
        f"word2vec_dim_{dimension}"
    )

    metadata_path = os.path.join(
        model_path,
        "metadata.json"
    )

    # Model đã tồn tại -> load
    if os.path.exists(model_path):
        print(f"Loading model: {model_path}")

        model = Word2VecModel.load(model_path)

        with open(metadata_path, "r") as f:
            metadata = json.load(f)

        train_time = metadata["train_time"]

        print(f"Original training time: {train_time:.2f}s")

        return model, train_time

    # Model chưa tồn tại -> train
    print(f"Training Word2Vec dimension = {dimension}")

    word2vec = Word2Vec(
        inputCol="words",
        outputCol="Embd",
        windowSize=5,
        vectorSize=dimension,
        minCount=5,
        numPartitions=8,
        maxIter=10,
        seed=42
    )

    start = time.perf_counter()

    model = word2vec.fit(words_df)

    elapsed = time.perf_counter() - start

    # Save model
    model.write().overwrite().save(model_path)

    # Save training time
    metadata = {
        "dimension": dimension,
        "train_time": elapsed
    }

    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=4)

    print(f"Training time: {elapsed:.2f}s")
    print(f"Saved model: {model_path}")

    return model, elapsed

In [10]:
tokenized_df = tokenized_df.cache()
tokenized_df.count()

30000

## Đánh giá thời gian chạy của các model
- Nếu chạy lần đầu cần train lại từ đầu
- Từ lần sau load lại model đã save

In [11]:
model_50, time_50 = create_model_dimension(dimension = 50, words_df = tokenized_df)

Loading model: /home/cong/26_8_learn_rag/NLP/lab3/models/word2vec_dim_50
Original training time: 192.25s


In [12]:
model_100, time_100 = create_model_dimension(dimension=100, words_df = tokenized_df)

Loading model: /home/cong/26_8_learn_rag/NLP/lab3/models/word2vec_dim_100
Original training time: 239.35s


In [13]:
model_300, time_300 = create_model_dimension(dimension=300, words_df=tokenized_df)

Loading model: /home/cong/26_8_learn_rag/NLP/lab3/models/word2vec_dim_300
Original training time: 353.00s


In [14]:
print("Training times:")
print(f"Dimension 50  : {time_50:.2f}s")
print(f"Dimension 100 : {time_100:.2f}s")
print(f"Dimension 300 : {time_300:.2f}s")

Training times:
Dimension 50  : 192.25s
Dimension 100 : 239.35s
Dimension 300 : 353.00s


## Đánh giá model size

In [15]:
def get_model_size(model):
    vocab_size = model.getVectors().count()
    dimension = len(
        model.getVectors().first()["vector"]
    )

    num_parameters = vocab_size * dimension
    size_bytes = num_parameters * 4
    size_mb = size_bytes / (1024 ** 2)

    return vocab_size, num_parameters, size_mb

In [16]:
vocab_50, params_50, size_50 = get_model_size(model_50)
vocab_100, params_100, size_100 = get_model_size(model_100)
vocab_300, params_300, size_300 = get_model_size(model_300)

In [17]:
print("Model size:")
print(
    f"Dim 50  : vocab={vocab_50}, "
    f"params={params_50:,}, size={size_50:.2f} MB"
)

print(
    f"Dim 100 : vocab={vocab_100}, "
    f"params={params_100:,}, size={size_100:.2f} MB"
)

print(
    f"Dim 300 : vocab={vocab_300}, "
    f"params={params_300:,}, size={size_300:.2f} MB"
)

Model size:
Dim 50  : vocab=52713, params=2,635,650, size=10.05 MB
Dim 100 : vocab=52713, params=5,271,300, size=20.11 MB
Dim 300 : vocab=52713, params=15,813,900, size=60.33 MB


## Đánh giá similarity

In [34]:
pairs = [
    ("doctor", "physician"),
    ("car", "automobile"),
    ("king", "queen"),
    ("cat", "dog"),
    ("computer", "banana")
]

similarity_results = []

for word1, word2 in pairs:

    similarity_results.append((
        f"{word1}-{word2}",
        word_similarity(model_50, word1, word2),
        word_similarity(model_100, word1, word2),
        word_similarity(model_300, word1, word2)
    ))

In [35]:
similarity_df = spark.createDataFrame(
    similarity_results,
    [
        "Word pair",
        "Dim 50",
        "Dim 100",
        "Dim 300"
    ]
)

similarity_df.show(truncate=False)

+----------------+--------------------+--------------------+---------------------+
|Word pair       |Dim 50              |Dim 100             |Dim 300              |
+----------------+--------------------+--------------------+---------------------+
|doctor-physician|0.841808432963989   |0.7551446726267516  |0.5542142350932383   |
|car-automobile  |0.7329757705516056  |0.6246224777721125  |0.4616538311061489   |
|king-queen      |0.8120568250725686  |0.7754111701628411  |0.6505745384065275   |
|cat-dog         |0.7386644892439297  |0.6266675218608748  |0.4804159959535884   |
|computer-banana |-0.23359629399538145|-0.16998167009173973|-0.048347882667127556|
+----------------+--------------------+--------------------+---------------------+



In [36]:
## xếp hạng similarity
def rank_similarity(model, word_pairs):

    results = []

    for word1, word2 in word_pairs:

        score = word_similarity(
            model,
            word1,
            word2
        )

        if score is not None:
            results.append(
                (
                    f"{word1} - {word2}",
                    score
                )
            )

    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return results

In [38]:
rank_50 = rank_similarity(
    model_50,
    pairs
)

rank_100 = rank_similarity(
    model_100,
    pairs

)

rank_300 = rank_similarity(
    model_300,
    pairs
)

In [51]:
import pandas as pd

pandas_df = pd.DataFrame({
    "Rank": list(range(1, len(rank_50) + 1)),
    "Dim 50 Pair": [p for p, s in rank_50],
    "Dim 50 Score": [s for p, s in rank_50],
    "Dim 100 Pair": [p for p, s in rank_100],
    "Dim 100 Score": [s for p, s in rank_100],
    "Dim 300 Pair": [p for p, s in rank_300],
    "Dim 300 Score": [s for p, s in rank_300],
})

df_ranks = spark.createDataFrame(pandas_df)

df_ranks.show()


+----+------------------+--------------------+------------------+--------------------+------------------+--------------------+
|Rank|       Dim 50 Pair|        Dim 50 Score|      Dim 100 Pair|       Dim 100 Score|      Dim 300 Pair|       Dim 300 Score|
+----+------------------+--------------------+------------------+--------------------+------------------+--------------------+
|   1|doctor - physician|   0.841808432963989|      king - queen|  0.7754111701628411|      king - queen|  0.6505745384065275|
|   2|      king - queen|  0.8120568250725686|doctor - physician|  0.7551446726267516|doctor - physician|  0.5542142350932383|
|   3|         cat - dog|  0.7386644892439297|         cat - dog|  0.6266675218608748|         cat - dog|  0.4804159959535884|
|   4|  car - automobile|  0.7329757705516056|  car - automobile|  0.6246224777721125|  car - automobile|  0.4616538311061489|
|   5| computer - banana|-0.23359629399538145| computer - banana|-0.16998167009173973| computer - banana|-0.048

/home/cong/miniconda3/envs/myenv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/cong/miniconda3/envs/myenv/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Đánh giá analogy

In [ ]:
def analogy(
    model,
    positive_words,
    negative_words,
    top_n=5
):

    vectors = model.getVectors()

    target = None

    for word in positive_words:

        vector = get_vector(
            model,
            word
        )

        if vector is None:
            return []

        if target is None:
            target = vector
        else:
            target = target + vector

    for word in negative_words:

        vector = get_vector(
            model,
            word
        )

        if vector is None:
            return []

        target = target - vector

    target_norm = float(
        target.norm(2)
    )

    results = []

    for row in vectors.collect():

        word = row["word"]

        if (
            word in positive_words
            or word in negative_words
        ):
            continue

        vector = row["vector"]

        similarity = float(
            target.dot(vector)
            / (
                target_norm
                * float(vector.norm(2))
            )
        )

        results.append(
            (word, similarity)
        )

    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return results[:top_n]

    

In [58]:
models = {
    "Dim 50": model_50,
    "Dim 100": model_100,
    "Dim 300": model_300
}

for name, model in models.items():
    print(f"\n===== {name} =====")

    results = analogy(
        model_50,
        positive_words=["king", "woman"],
        negative_words=["man"],
        top_n=5
        )

    for rank, (word, score) in enumerate(results, start=1):
        print(
            f"{rank}. {word:<15} "
            f"similarity={score:.4f}"
        )


===== Dim 50 =====
1. queen           similarity=0.7626
2. paulette        similarity=0.7022
3. madison         similarity=0.6947
4. regal           similarity=0.6805
5. decorates       similarity=0.6754

===== Dim 100 =====
1. queen           similarity=0.7626
2. paulette        similarity=0.7022
3. madison         similarity=0.6947
4. regal           similarity=0.6805
5. decorates       similarity=0.6754

===== Dim 300 =====
1. queen           similarity=0.7626
2. paulette        similarity=0.7022
3. madison         similarity=0.6947
4. regal           similarity=0.6805
5. decorates       similarity=0.6754


In [59]:
models = {
    "Dim 50": model_50,
    "Dim 100": model_100,
    "Dim 300": model_300
}

for name, model in models.items():
    print(f"\n===== {name} =====")

    results = analogy(
        model,
        positive_words=["paris", "italy"],
        negative_words=["france"],
        top_n=5
    )

    for rank, (word, score) in enumerate(results, start=1):
        print(
            f"{rank}. {word:<15} "
            f"similarity={score:.4f}"
        )


===== Dim 50 =====
1. switzerland     similarity=0.7965
2. copenhagen      similarity=0.7716
3. denmark         similarity=0.7656
4. 1630            similarity=0.7636
5. austria         similarity=0.7591

===== Dim 100 =====
1. switzerland     similarity=0.6642
2. copenhagen      similarity=0.6128
3. venice          similarity=0.6072
4. monaco          similarity=0.6071
5. austria         similarity=0.6049

===== Dim 300 =====
1. rome            similarity=0.3777
2. arte            similarity=0.3730
3. oktoberfest     similarity=0.3719
4. turin           similarity=0.3708
5. austria         similarity=0.3686


### Kết luận: Ta có thể thấy với model dimension tuy rằng thời gian tính toán lâu hơn, lưu trữ nặng hơn nhưng khả năng bắt ngữ nghĩa tốt hơn đã bắt được quan hệ của paris italy france điều mà 2 model 100 và model 50 đã thất bại

## Đánh giá AUC dựa trên downstream task - sentiment classification
Ở phần Donwstream task này em chọn task là Sentiment Classification của dữ liệu IDBM
- Ở đây em train các model lần lượt với vector size là 50, 100, 300 trên tập dữ liệu 30k 
- Sau đó em tiến hành task sentiment classification trên tập dữ liệu IDBM classification và đánh giá bằng metrics AUC để kiểm tra performance của mô hình logistic regression với các model Embedding Dimension lần lượt là 50, 100, 300

In [28]:
path = "/home/cong/26_8_learn_rag/NLP/lab3/data_imdb/imdb/IMDB Dataset.csv"

orginal = spark.read.csv(path, header=True, multiLine=True, escape='"')

clean_df = orginal.filter(orginal.sentiment.isin(["positive", "negative"]))

from pyspark.ml.feature import RegexTokenizer

tokenizer = RegexTokenizer(
    inputCol="review",
    outputCol="words",
    pattern=r"[\S.,?!:;\"/)(><\\]+",
    gaps=False
)

# SỬA Ở ĐÂY: Dùng clean_df thay vì orginal
words_df = tokenizer.transform(clean_df)
words_df.show(5)

from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="sentiment",
    outputCol="label",
    handleInvalid="skip"
)

indexer_model = indexer.fit(words_df)

df_index = indexer_model.transform(words_df)

df_index.select("sentiment", "label").show(5)

train_set, test_set = df_index.randomSplit([0.8, 0.2], seed=42)

train_set.show(5)


+--------------------+---------+--------------------+
|              review|sentiment|               words|
+--------------------+---------+--------------------+
|One of the other ...| positive|[one, of, the, ot...|
|A wonderful littl...| positive|[a, wonderful, li...|
|I thought this wa...| positive|[i, thought, this...|
|Basically there's...| negative|[basically, there...|
|Petter Mattei's "...| positive|[petter, mattei's...|
+--------------------+---------+--------------------+
only showing top 5 rows
+---------+-----+
|sentiment|label|
+---------+-----+
| positive|  1.0|
| positive|  1.0|
| positive|  1.0|
| negative|  0.0|
| positive|  1.0|
+---------+-----+
only showing top 5 rows


+--------------------+---------+--------------------+-----+
|              review|sentiment|               words|label|
+--------------------+---------+--------------------+-----+
|\b\b\b\bA Turkish...| positive|[\b\b\b\ba, turki...|  1.0|
|!!! Spoiler alert...| negative|[!!!, spoiler, al...|  0.0|
|!!!! MILD SPOILER...| negative|[!!!!, mild, spoi...|  0.0|
|!!!! POSSIBLE MIL...| negative|[!!!!, possible, ...|  0.0|
|!!!!! OF COURSE T...| negative|[!!!!!, of, cours...|  0.0|
+--------------------+---------+--------------------+-----+
only showing top 5 rows


In [29]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

def downstream_task(dimension, model, df_train, df_test):
    print("DOWNSREAM TASK Model: ", dimension)
    emb_df_train = model.transform(df_train)
    emb_df_test = model.transform(df_test)

    lr = LogisticRegression(
        featuresCol = "Embd",
        labelCol = "label",
    )

    cls = lr.fit(emb_df_train)
    
    predictions = cls.transform(emb_df_test)

    
    evaluator = BinaryClassificationEvaluator(
            labelCol="label",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC" # "areaUnderROC" chính là AUC
    )
    auc = evaluator.evaluate(predictions)
    print(f"Model using dimension {dimension} - AUC (Area Under ROC): {auc:.4f}")
    return auc


In [30]:
auc_50 = downstream_task(50, model_50, train_set, test_set)

DOWNSREAM TASK Model:  50


Model using dimension 50 - AUC (Area Under ROC): 0.8220


In [31]:
auc_100 = downstream_task(100, model_100, train_set, test_set)

DOWNSREAM TASK Model:  100


Model using dimension 100 - AUC (Area Under ROC): 0.8524


In [32]:
auc_300 = downstream_task(300, model_300, train_set, test_set)

DOWNSREAM TASK Model:  300


Model using dimension 300 - AUC (Area Under ROC): 0.8836


In [33]:
print("Performance of model on AUC metrics")
print("Model dimension 50 ", auc_50)
print("Model dimension 100 ", auc_100)
print("Model dimension 200 ", auc_300)

Performance of model on AUC metrics
Model dimension 50  0.822029239977975
Model dimension 100  0.8523901451544593
Model dimension 200  0.8836006408013708
